# T8: Strings: A Deeper Look
CHE-226, Programming and Data Science

**Learning objective:** format numbers into readable reports, clean, compare, search, split, and join text, and use simple regular expressions to validate and extract values from instrument tags and log lines.

**Teaches:** (KA-01)

*Spans two class meetings; see the class break marker partway through.*

## Agenda: This Class
1. Formatting numbers in f-strings
2. Field widths, alignment, and the format method
3. Stripping whitespace and changing case
4. Comparing strings
5. Searching for substrings

# 1. Formatting Numbers in f-strings

### Concept
After the colon in `{value:spec}`, a presentation type controls how a number looks: `f` fixed-point, `e` scientific, `%` percent, `d` integer, with `,` for thousands and `+` for a sign.

In [ ]:
k = 0.000197          # rate constant, 1/s
Q = 1234567.891       # heat duty, W
x = 0.4375            # conversion
n = 7                 # number of trays

print(f"{k:.2e}  {Q:,.1f}  {x:.1%}  {n:d}  {n:+d}")

Detailed notes: The format spec mini-language is: fill and alignment, sign, width, grouping, precision, then type. .2f means two digits after the decimal point; .2e means two digits after the point in scientific notation, the right choice for very small or very large engineering values such as rate constants; .1% multiplies by 100 and adds the percent sign; , inserts thousands separators; + forces a sign even for positive numbers, useful for deviations from a setpoint. d works only on integers and raises ValueError for a float. Strings use s, and c turns an integer character code into its character. These are formatting choices only; the stored value is unchanged.

### Activity: predict then run
What does each field print?

In [ ]:
print(f"{0.00321:.1e} | {0.875:.0%} | {-2.5:+.1f} | {1500000:,}")

Answer: 3.2e-03 | 88% | -2.5 | 1,500,000. 0.875 as .0% is 87.5 percent, rounded to 88; the + spec keeps the existing minus sign; a plain comma spec groups an integer's digits.

# 2. Field Widths, Alignment, and the format Method

### Concept
A width reserves that many characters; `<`, `>`, and `^` align left, right, and center within it. The older `str.format` method fills `{}` placeholders by position or name.

In [ ]:
streams = [("Feed", 350.0, 5.0), ("Product", 412.3, 1.2)]

print(f"{'Stream':<10}{'T (K)':>8}{'P (bar)':>9}")
for name, T, P in streams:
    print(f"{name:<10}{T:>8.1f}{P:>9.2f}")
print(f"{'END':-^27}")                          # centered, '-' fill

print("{} at {:.1f} K".format("Feed", 350.0))    # format method

Detailed notes: Numbers align right by default and strings left, but setting the alignment explicitly makes a table's intent obvious. A fill character goes before the alignment symbol, as in {'END':-^27}. Fixed widths are what turn a column of printed numbers into a readable stream table; this is the same report T5 built, now with aligned columns. str.format predates f-strings and still appears in older code and libraries; placeholders can be numbered ({0}, {1}) or named ({name}), and the same format specs work after the colon. + joins strings and * repeats them ('-' * 27), which is another way to draw a rule line.

### Activity: predict then run
What does each line print? Count the characters.

In [ ]:
print("{1} before {0}".format("A", "B"))
print(f"[{'CO2':>6}]")
print("=" * 3 + "ab" * 2)

Answer: B before A (placeholders by position); [   CO2] (three spaces, then CO2 right-aligned in 6); ===abab.

# 3. Stripping Whitespace and Changing Case

### Concept
`strip`, `lstrip`, and `rstrip` remove surrounding whitespace. `lower`, `upper`, `capitalize`, and `title` change case. Strings are immutable, so each returns a new string.

In [ ]:
raw = "  Ethanol \n"             # as typed or read from a file
name = raw.strip()               # remove spaces and the newline
print(repr(raw), repr(name))
print(name.lower(), name.upper())
print("heat exchanger e-101".title())
print("xylene".capitalize())

Detailed notes: Text arriving from a keyboard, a file, or an instrument usually carries stray spaces, tabs, and a trailing newline. repr shows those invisible characters, which is the fastest way to debug a comparison that should match but does not. strip also accepts characters to remove, for example strip('*'). Normalizing case before comparing ('Ethanol', 'ETHANOL', 'ethanol') is standard practice for user input; title capitalizes every word and capitalize only the first letter of the whole string. Because strings are immutable, none of these methods changes the original; forgetting to assign the result is the bug in the activity.

### Activity: spot the bug
The user typed `" Water"`. This should print `Found water`, but it does not. Why?

In [ ]:
component = " Water"
component.strip()
component.lower()
if component == "water":
    print("Found water")
else:
    print("Not found:", repr(component))

Answer: prints Not found: ' Water'. strip and lower return new strings, and the results are discarded, so component is unchanged. Fix: component = component.strip().lower(). Chaining the calls on one line is idiomatic.

# 4. Comparing Strings

### Concept
`==` compares strings exactly, case included. `<` and `>` compare character by character using Unicode code points, so uppercase letters sort before lowercase ones.

In [ ]:
print("water" == "Water")              # case matters
print(ord("A"), ord("Z"), ord("a"))    # Unicode code points
print("Benzene" < "Toluene")           # 'B' (66) < 'T' (84)
print(sorted(["toluene", "Xylene", "benzene"]))
print(sorted(["toluene", "Xylene", "benzene"], key=str.lower))

Detailed notes: String comparison is lexicographic: compare the first characters; if equal, compare the second, and so on; a string that runs out first is smaller. Each character is compared by its code point (ord), and every uppercase letter (65 to 90) comes before every lowercase letter (97 to 122). That is why sorting mixed-case names looks wrong until you pass key=str.lower. Digits (48 to 57) come before letters, and comparison is character by character rather than numeric, which produces the surprise in the activity: tags or file names with numbers sort in a way that is not numeric.

### Activity: predict then run
True or False for each?

In [ ]:
print("Z" < "a")
print("10" < "9")
print("R-10" < "R-9")

Answer: True, True, True. 'Z' is 90 and 'a' is 97. '10' < '9' because '1' (49) is less than '9' (57), and the comparison never gets to the second character. So tag R-10 sorts before R-9; convert the number part with int() before sorting, or zero-pad the tags (R-09).

# 5. Searching for Substrings

### Concept
`in` tests for a substring. `count`, `find`, and `index` locate it; `find` returns -1 if it is missing, `index` raises an error. `startswith` and `endswith` check the ends.

In [ ]:
tag = "TIC-101A"                        # temperature controller

print("IC" in tag)                      # substring test
print(tag.count("1"))                   # non-overlapping occurrences
print(tag.find("-"), tag.find("X"))     # position, or -1
print(tag.rfind("1"))                   # search from the right
print(tag.startswith("T"), tag.endswith(("A", "B")))

Detailed notes: In ISA instrument tags the first letters encode what is measured and what the device does (T for temperature, I for indicate, C for control), so text searches like these are how a script sorts a tag list by instrument type. find is safe inside an if; index is right when a missing substring really is an error. Both accept optional start and end positions. startswith and endswith accept a tuple of options, as in the last line. Positions are zero-based, exactly like list indices, and slicing works on strings: tag[:3] is 'TIC'.

### Activity: predict then run
Using `tag = "TIC-101A"`, what does each call return?

In [ ]:
print(tag.find("1"), tag.index("C"), tag[4:7])
print(tag.lower().startswith("tic"), tag.count("I"))

Answer: 4 2 101, then True 1. find returns the first match at index 4; C is at index 2; the slice 4:7 is the loop number. After lower(), the tag starts with 'tic'.

## Class Break
Covered so far: number formatting, field widths and alignment, stripping and case, comparing strings, searching.

## Agenda: Next Class
6. Replacing, splitting, and joining
7. partition, splitlines, and character tests
8. Regular expressions: matching patterns
9. Regular expressions: extracting data
10. Putting it together: cleaning a lab log

# 6. Replacing, Splitting, and Joining

### Concept
`replace` swaps every occurrence of a substring. `split` breaks a string into a list at a separator; `join` glues a list of strings back together.

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
line = "H2O:0.70, EtOH:0.25, MeOH:0.05"     # composition as text

parts = line.replace(" ", "").split(",")   # ['H2O:0.70', ...]
comp = {}
for p in parts:
    name, frac = p.split(":")
    comp[name] = float(frac)               # text to number
print(comp)
print(" + ".join(comp))                    # join the keys

Detailed notes: Parsing text into numbers is the everyday use: split on the field separator, split each field on the key-value separator, convert with float, and store in a dictionary (T6). split() with no argument splits on any run of whitespace and drops empty strings, which is usually what you want for space-separated data; split(',') splits on every comma, keeping empty fields. maxsplit limits the number of splits. join is called on the separator, not the list, and every item must already be a string; ', '.join(map(str, numbers)) handles numbers. replace returns a new string, as always.

### Activity: predict then run
What does each split return?

In [ ]:
print("350,,352".split(","))
print("  350   352 ".split())
print("a:b:c".split(":", 1))

Answer: ['350', '', '352'] (an explicit separator keeps the empty field, a missing reading); ['350', '352'] (whitespace splitting collapses runs and trims); ['a', 'b:c'] (maxsplit of 1).

# 7. partition, splitlines, and Character Tests

### Concept
`partition` splits once into (before, separator, after). `splitlines` splits multi-line text into lines. Methods like `isdigit` and `isalpha` test what characters a string holds.

In [ ]:
reading = "T=350.2"
key, sep, value = reading.partition("=")   # always 3 parts
print(key, value)

block = "FIC-101\nTIC-205\nLIC-310"
print(block.splitlines())

print("101".isdigit(), "TIC".isalpha(), "TIC101".isalnum())

Detailed notes: partition is split(sep, 1) that always returns exactly three parts, so unpacking into three names never fails even when the separator is missing (sep and the last part are then empty strings). rpartition splits at the last occurrence instead, handy for file extensions. splitlines handles \n, \r\n, and other line endings, so it is safer than split('\n') on files from different operating systems. isdigit, isalpha, isalnum, isspace, isupper, and islower all return False for an empty string. Warn students that these tests look at characters, not at whether a string could be converted to a number, which the activity shows.

### Activity: predict then run
Would each of these pass an `isdigit()` check?

In [ ]:
print("350".isdigit(), "350.2".isdigit(), "-5".isdigit())

Answer: True False False. The decimal point and the minus sign are not digits, so isdigit is a poor test for numeric input. The robust way to ask whether text is a number is to try float(text) and catch ValueError, which T9 covers.

# 8. Regular Expressions: Matching Patterns

### Concept
A regular expression describes a text pattern. `re.fullmatch` checks whether a whole string fits it. Write patterns as raw strings `r"..."` so backslashes stay literal.

In [ ]:
import re

pattern = r"[A-Z]{2,3}-\d{3}"      # 2-3 capitals, '-', 3 digits
for tag in ["TIC-101", "PI-20", "fic-101", "LAHH-310"]:
    ok = re.fullmatch(pattern, tag)  # a match object, or None
    print(f"{tag:<9} {'valid' if ok else 'invalid'}")

Detailed notes: A raw string, r'...', tells Python not to treat backslashes as escape sequences, so r'\d' reaches the re module intact; without it, some patterns silently break. The building blocks: character classes such as \d (digit), \w (word character), \s (whitespace), and [A-Z] (a custom range) or [^...] (anything except); quantifiers such as * (zero or more), + (one or more), ? (optional), and {n,m} (between n and m). Special characters such as . or - outside a class need a backslash to be matched literally. fullmatch requires the entire string to match; re.search finds a match anywhere. Regular expressions are the standard tool for validating formatted input such as tag numbers, dates, or chemical formulas.

### Activity: predict then run
The pattern is `r"[A-Z]{2,3}-\d{3}"`. Which of these match?

In [ ]:
for t in ["TI-100", "TIC-1000", "T-100", "PDI-007"]:
    print(t, bool(re.fullmatch(pattern, t)))

Answer: TI-100 True; TIC-1000 False (four digits, and fullmatch needs the whole string); T-100 False (only one letter); PDI-007 True (leading zeros are still digits).

# 9. Regular Expressions: Extracting Data

### Concept
`re.findall` returns every match in a string; `re.sub` replaces matches; `re.split` splits on a pattern. Parentheses in a pattern capture groups you can read back.

In [ ]:
line = "2026-10-05 08:15 T=350.2K P=5.10bar F=12.5kg/s"

nums = re.findall(r"\d+\.\d+", line)          # decimal numbers
print(nums)
m = re.search(r"T=(\d+\.?\d*)K", line)        # capture group 1
print(float(m.group(1)))
print(re.sub(r"\s+", " ", "T =   350   K"))    # squeeze spaces
print(re.split(r"[ =]", "T=350 P=5"))

Detailed notes: findall with no groups returns a list of matched strings; with one group it returns just the captured parts. search returns a match object for the first hit, or None, so check before calling group. group(0) is the whole match and group(1) the first parenthesized part. re.IGNORECASE (flags=re.I) makes a pattern case-insensitive, and ^ and $ anchor a match to the start and end. finditer is the memory-friendly version of findall for very large text. A pattern should be as specific as the data require: a pattern that is too loose matches things it should not, and one that is too strict silently skips real data, which is the bug in the activity.

### Activity: spot the bug
This should pull the temperature and pressure out of the line as two numbers. It runs, but returns the wrong list. Why?

In [ ]:
line = "T=350.2K P=5.10bar"
values = re.findall(r"\d+", line)
print(values)

Answer: prints ['350', '2', '5', '10']. \d+ matches runs of digits only, so each decimal point splits a number in two. Fix: r'\d+\.?\d*' (digits, an optional point, more digits), which gives ['350.2', '5.10'].

# 10. Putting It Together: Cleaning a Lab Log

### Concept
Real log files mix comments, blank lines, messy spacing, and units. Strip, skip, split, and convert each line, then report with aligned f-strings.

In [ ]:
log = """# batch 17, reactor R-101
  08:00 , 350.2 K
08:15,351.9 K

08:30 , 355.0 K"""
temps = []
for line in log.splitlines():
    if not line.strip() or line.startswith("#"):
        continue                                # skip blank/comment
    value = line.split(",")[1]                  # '350.2 K'
    temps.append(float(value.split()[0]))       # drop the unit
print(temps, f"mean = {sum(temps) / len(temps):.1f} K")

Detailed notes: This combines the lecture on one realistic chore: splitlines breaks the text into lines, strip detects blank lines and removes stray spaces around the value, an if with continue (from T3) skips blank lines and comments, split separates time from value and the value from its unit, and float converts. The time field is ignored here but would be kept in a real script. It is data cleaning, the unglamorous step that takes most of the time in real data science. T9 reads the same kind of text from an actual file, and pandas then does much of this work in one call for well-formed tables.

### Activity: cold call
Suppose one line reads `08:45, ---- K` because the sensor dropped out. What happens when the loop reaches it, and what would you change?

In [ ]:
float("----")

Answer: float('----') raises ValueError and the whole loop stops, losing every later reading. Reasonable fixes: check the value with a regex before converting, or wrap the conversion in try/except and skip bad lines, which is the first topic of T9.

## Recap and Lab Practice
- Format specs after the colon (.2f, .2e, .1%, ',', widths, < > ^) turn numbers into readable reports.
- Strings are immutable: strip, lower, replace, and friends return new strings you must assign. Comparison is by character code, so '10' < '9'.
- split, join, and partition parse structured text; regular expressions validate and extract patterns such as tags and numbers.

Lab practice: parse a composition string into a dictionary, validate a list of instrument tags with a regex, and clean a messy lab log into a formatted report; ask if you want them turned into a lab worksheet.

Next lecture: T9, files and exceptions.